# MNIST 三层 MLP（输入层+隐藏层+输出层）

这个 Notebook 与 `train_mnist_mlp.py` 一致：
- 总层数 = 3（输入、隐藏、输出）
- 自动选择设备：MPS -> CUDA -> CPU
- 保存最佳模型

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from pathlib import Path

In [ ]:
def get_device():
    if torch.backends.mps.is_built() and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

device = get_device()
print("using device:", device)

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden_dim=256, dropout=0.1):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 10),
        )

    def forward(self, x):
        return self.layers(x)

In [ ]:
# 调参入口
BATCH_SIZE = 128
EPOCHS = 12
LR = 8e-4
HIDDEN_DIM = 256
DROPOUT = 0.1
WEIGHT_DECAY = 1e-4
DATA_DIR = Path("./data")
SAVE_PATH = Path("./best_mlp_mnist.pth")

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_set = datasets.MNIST(root=DATA_DIR, train=True, download=True, transform=transform)
test_set = datasets.MNIST(root=DATA_DIR, train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

print("train samples:", len(train_set), "| test samples:", len(test_set))

In [ ]:
model = MLP(hidden_dim=HIDDEN_DIM, dropout=DROPOUT).to(device)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss()

In [ ]:
def train_one_epoch(model, device, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for batch_idx, (data, target) in enumerate(loader):
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        logits = model(data)
        loss = criterion(logits, target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        total_correct += (preds == target).sum().item()
        total_count += target.size(0)

        if batch_idx % 100 == 0:
            print(f"batch {batch_idx:4d}/{len(loader):4d} | loss {loss.item():.4f}")

    avg_loss = total_loss / len(loader)
    acc = 100.0 * total_correct / total_count
    return avg_loss, acc

@torch.no_grad()
def evaluate(model, device, loader):
    model.eval()
    total_correct = 0
    total_count = 0

    for data, target in loader:
        data, target = data.to(device), target.to(device)
        logits = model(data)
        preds = logits.argmax(dim=1)
        total_correct += (preds == target).sum().item()
        total_count += target.size(0)

    return 100.0 * total_correct / total_count

In [ ]:
best_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    print(f"\nepoch {epoch}/{EPOCHS}")
    print("-" * 30)

    train_loss, train_acc = train_one_epoch(model, device, train_loader, optimizer, criterion)
    test_acc = evaluate(model, device, test_loader)

    print(f"train loss: {train_loss:.4f} | train acc: {train_acc:.2f}% | test acc: {test_acc:.2f}%")

    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"saved best model to: {SAVE_PATH} (acc: {best_acc:.2f}%)")

print("\ndone. best test accuracy:", round(best_acc, 2))